## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-13-49-37 +0000,startribune,Amy Klobuchar and Peggy Flanagan triumph as De...,https://www.startribune.com/live-minnesota-202...
1,2026-11-04-13-49-00 +0000,wsj,Meta Seeks to Raise $9.2 Billion in Debut Euro...,https://www.wsj.com/business/meta-seeks-to-rai...
2,2026-11-04-13-49-00 +0000,wsj,Opinion | Will Democrats Misread Their Victory?,https://www.wsj.com/opinion/democrats-2026-ele...
3,2026-11-04-13-48-00 +0000,wsj,Empower Brands Makes Unsolicited Bid to Buy He...,https://www.wsj.com/business/deals/empower-bra...
4,2026-11-04-13-46-00 +0000,wsj,Oil Prices Fall as Investors Weigh U.S. Midter...,https://www.wsj.com/business/energy-oil/oil-pr...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2476/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
52,trump,54
186,race,47
18,democrats,41
48,election,39
35,midterm,31


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
336,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,258
74,2026-11-04-11-34-05 +0000,chitri,Democrats win the US House and Senate as voter...,https://www.chicagotribune.com/2026/11/04/poll...,218
229,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...,217
104,2026-11-04-10-30-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...,202
264,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,201


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
336,258,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
229,217,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...
23,112,2026-11-04-12-54-34 +0000,cbc,Iranian officials welcome U.S. midterm results...,https://www.cbc.ca/news/world/iran-united-stat...
234,102,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
36,75,2026-11-04-12-36-00 +0000,wsj,Americans’ long-simmering frustrations over th...,https://www.wsj.com/politics/elections/were-ju...
356,67,2026-11-03-22-01-05 +0000,nyt,How Will the Midterms Play Out? These Races Ma...,https://www.nytimes.com/2026/11/03/us/politics...
297,67,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...
272,65,2026-11-04-02-35-53 +0000,chitri,Georgia governor race: Democrat Keisha Lance B...,https://www.chicagotribune.com/2026/11/03/geor...
104,65,2026-11-04-10-30-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...
141,49,2026-11-04-07-32-02 +0000,bbc,South Carolina elects Democratic senator for f...,https://www.bbc.co.uk/news/articles/cjy56z440n...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
